# 股票池优化：跨 Universe 简易实验

目标：在尽量不改动既有 `uncle_stock/机器学习策略` 口径的前提下，比较不同股票池是否能给当前机器学习选股模型提供更好的 alpha 来源。

实验口径：

- 使用当前核心观察包里的 V4/V46 风格特征构造。
- 默认加载 `model_candidate_v46_lgb_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl`。
- 每月第一个交易日调仓，使用前一个交易日特征。
- 每个股票池用同一个模型打分、同一个 `top30 -> 行业分散 top10` 组合规则。
- 评价持有到下一个月度调仓日的等权收益、相对中证800的超额收益、RankIC、胜率、回撤、换手和流动性。

注意：这不是最终实盘回测，而是 universe 研究实验。它回答的是“当前模型信号迁移到哪个股票池更有效”。如果某个扩展池显著更好，下一步才值得对该池重建训练集并重训模型。

In [ ]:
# =========================
# Cell 1: Imports & Config
# =========================
import gc
import os
import pickle
import datetime
import warnings

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception as err:
    print("JoinQuant API import skipped. This notebook should be run in JQ research/runtime.", err)

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

# ---- Core experiment knobs ----
MODEL_FILE = "model_candidate_v46_lgb_direct_hybrid_l2_ff10_2019_2025q1_legacy_unsealed.pkl"
EXPERIMENT_START = "2022-01-01"
EXPERIMENT_END_FOR_LABEL = "2026-05-31"
BENCHMARK = "000906.XSHG"  # 中证800，沿用当前策略 alpha label 口径
MIN_LISTING_DAYS = 180

TOP_N_CANDIDATES = 30
TOP_N_PORTFOLIO = 10
INDUSTRY_CAP_RATIO = 0.20

# 扩展池的流动性门槛。想更偏实盘可提高到 1e8，想探索 alpha 可降到 3e7。
LIQ_MIN_MONEY_20 = 50_000_000
LIQ_MAX_PAUSED_20 = 3

# 为了先快跑，建议先保留 4 个核心池；确认 OK 后再打开 ALL_A_LIQ。
POOL_CONFIGS = [
    {"name": "CSI300", "type": "index", "index": "000300.XSHG", "liquidity_filter": False},
    {"name": "CSI800", "type": "index", "index": "000906.XSHG", "liquidity_filter": False},
    {"name": "CSI1000", "type": "index", "index": "000852.XSHG", "liquidity_filter": False},
    {"name": "CSI800_PLUS_1000_LIQ", "type": "union", "indexes": ["000906.XSHG", "000852.XSHG"], "liquidity_filter": True},
    # 创业板卫星池：从全A中取 30 开头股票，再加流动性约束。
    {"name": "CHINEXT_LIQ", "type": "prefix_from_index", "index": "000985.XSHG", "prefixes": ["30"], "liquidity_filter": True},
    # 小盘卫星池：全A流动性过滤后按当期市值取最小的1000只。
    {"name": "SMALLCAP_LIQ_1000", "type": "index", "index": "000985.XSHG", "liquidity_filter": True, "smallcap_limit": 1000},
    # 全A流动性池较慢，首轮可以注释掉；需要更完整比较时打开。
    {"name": "ALL_A_LIQ", "type": "index", "index": "000985.XSHG", "liquidity_filter": True},
]

CACHE_DIR = "universe_pool_lab_cache"
FORCE_REBUILD = False
os.makedirs(CACHE_DIR, exist_ok=True)

print("config ready")

In [ ]:
# =========================
# Cell 2: Feature constants from 机器学习策略 V4/V46口径
# =========================
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield",
    "sales_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit", "ACCA", "growth", "net_working_capital",
    "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV",
    "debt_to_equity_ratio", "debt_to_tangible_equity_ratio", "momentum",
    "Rank1M", "sharpe_ratio_60", "Variance20", "liquidity", "beta",
    "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

V4_PRICE_PATH_COLS = [
    "px_ret_5", "px_ret_20", "px_ret_60", "px_ret_120",
    "px_close_to_ma20", "px_close_to_ma60", "px_ma20_to_ma60",
    "px_volatility_20", "px_volatility_60", "px_drawdown_20",
    "px_drawdown_60", "px_drawdown_120", "px_up_day_ratio_20",
    "px_new_high_distance_60", "px_new_low_distance_60", "px_skew_20", "px_kurt_20",
]

TRADE_LIQUIDITY_COLS = [
    "liq_money_mean_20", "liq_money_mean_60", "liq_money_ratio_20_60",
    "liq_volume_mean_20", "liq_volume_ratio_20_60", "liq_amplitude_mean_20",
    "liq_amplitude_mean_60", "liq_paused_count_20", "liq_paused_count_60",
    "liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20",
    "liq_one_price_limit_count_20",
]

CONTEXT_COLS = [
    "ctx_industry_ret_20", "ctx_industry_ret_60", "ctx_stock_minus_industry_ret_20",
    "ctx_stock_minus_industry_ret_60", "ctx_stock_rank_industry_ret_20",
    "ctx_stock_rank_industry_volatility_20", "ctx_market_ret_20",
    "ctx_market_ret_60", "ctx_market_volatility_20",
]

CORE_TEMPORAL_FACTORS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio",
    "Rank1M", "sharpe_ratio_60", "VOSC", "MFI14",
]

INDUSTRY_RELATIVE_FACTORS = [
    "book_to_price_ratio", "earnings_yield", "cash_flow_to_price_ratio",
    "roe_ttm", "roa_ttm", "Rank1M", "sharpe_ratio_60",
]

print("feature constants ready")

In [ ]:
# =========================
# Cell 3: Generic helpers
# =========================
def unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 5 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def zscore_series(s):
    s = pd.Series(s).astype(float)
    std = s.std()
    if pd.isnull(std) or std <= 0:
        return s * 0.0
    return (s - s.mean()) / std


def require_joinquant_api():
    names = ["get_trade_days", "get_index_stocks", "get_factor_values", "get_price", "get_security_info", "get_extras"]
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError("missing JoinQuant APIs: {}".format(",".join(missing)))


def get_period_date(period, start_date, end_date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    if len(trade_days) == 0:
        return []
    if period != "M":
        raise ValueError("only monthly period M is supported")
    dates = []
    last_key = None
    for d in trade_days:
        key = d.strftime("%Y-%m")
        if key != last_key:
            dates.append(d.strftime("%Y-%m-%d"))
            last_key = key
    return dates


def get_previous_trade_date(date):
    require_joinquant_api()
    trade_days = pd.to_datetime(get_trade_days(end_date=date, count=2))
    if len(trade_days) < 2:
        return None
    return trade_days[-2].strftime("%Y-%m-%d")


def max_drawdown(monthly_ret):
    r = pd.Series(monthly_ret).dropna().astype(float)
    if r.empty:
        return np.nan
    nav = (1.0 + r).cumprod()
    dd = nav / nav.cummax() - 1.0
    return dd.min()

print("helpers ready")

In [ ]:
# =========================
# Cell 4: Universe and filters
# =========================
def get_raw_pool(pool_config, feature_date):
    if pool_config["type"] == "index":
        return list(get_index_stocks(pool_config["index"], feature_date))
    if pool_config["type"] == "union":
        out = []
        for idx in pool_config["indexes"]:
            out.extend(list(get_index_stocks(idx, feature_date)))
        return unique_keep_order(out)
    if pool_config["type"] == "prefix_from_index":
        base = list(get_index_stocks(pool_config["index"], feature_date))
        prefixes = tuple(pool_config.get("prefixes", []))
        return [s for s in base if s.startswith(prefixes)]
    raise ValueError("unsupported pool config: {}".format(pool_config))


def filter_st_by_date(stock_list, feature_date):
    if len(stock_list) == 0:
        return []
    try:
        st_data = get_extras("is_st", stock_list, count=1, end_date=feature_date)
    except Exception:
        st_data = None
    if st_data is None or len(st_data) == 0:
        return stock_list
    row = st_data.iloc[0]
    return [s for s in stock_list if (s not in row.index) or pd.isnull(row[s]) or (not bool(row[s]))]


def filter_paused_by_date(stock_list, feature_date):
    if len(stock_list) == 0:
        return []
    try:
        paused_df = get_price(
            stock_list, end_date=feature_date, frequency="daily", fields=["paused"],
            count=1, skip_paused=False, panel=False, fill_paused=True,
        )
    except Exception:
        return stock_list
    if paused_df is None or paused_df.empty or "paused" not in paused_df.columns:
        return stock_list
    paused_map = paused_df.groupby("code")["paused"].last()
    return [s for s in stock_list if (s not in paused_map.index) or (not bool(paused_map.loc[s]))]


def filter_min_listing_days(stock_list, feature_date, min_days=MIN_LISTING_DAYS):
    out = []
    begin_dt = pd.Timestamp(feature_date).to_pydatetime()
    for s in stock_list:
        try:
            info = get_security_info(s)
        except Exception:
            info = None
        if info is None:
            continue
        if info.start_date <= (begin_dt - datetime.timedelta(days=min_days)).date():
            out.append(s)
    return out


def filter_smallcap_by_market_cap(stock_list, feature_date, limit):
    if len(stock_list) == 0 or limit is None:
        return stock_list
    limit = int(limit)
    if len(stock_list) <= limit:
        return stock_list
    rows = []
    for stock_chunk in chunks(stock_list, 500):
        try:
            df = get_fundamentals(
                query(valuation.code, valuation.market_cap).filter(valuation.code.in_(stock_chunk)),
                date=feature_date,
            )
        except Exception as err:
            print("market cap query failed", feature_date, len(stock_chunk), err)
            df = None
        if df is not None and not df.empty:
            rows.append(df)
    if not rows:
        return stock_list[:limit]
    cap = pd.concat(rows, ignore_index=True).dropna(subset=["market_cap"])
    cap = cap.sort_values(["market_cap", "code"], ascending=[True, True])
    selected = cap.head(limit)["code"].tolist()
    selected_set = set(selected)
    return [s for s in stock_list if s in selected_set]


def get_prefiltered_pool(pool_config, feature_date):
    stock_list = get_raw_pool(pool_config, feature_date)
    stock_list = filter_st_by_date(stock_list, feature_date)
    stock_list = filter_paused_by_date(stock_list, feature_date)
    stock_list = filter_min_listing_days(stock_list, feature_date, MIN_LISTING_DAYS)
    return unique_keep_order(stock_list)

print("universe helpers ready")

In [ ]:
# =========================
# Cell 5: Feature builders, adapted from 机器学习策略/research_scripts/candidate_model_export.py
# =========================
def get_industry_bucket_map(stock_list, date):
    if len(stock_list) == 0:
        return {}
    try:
        industry_info = get_industry(stock_list, date=date)
    except Exception:
        return {s: "UNKNOWN" for s in stock_list}
    out = {}
    for s in stock_list:
        info = industry_info.get(s, {})
        bucket = None
        for key in ["sw_l1", "jq_l1", "zjw"]:
            sub = info.get(key, None)
            if isinstance(sub, dict):
                bucket = sub.get("industry_code") or sub.get("industry_name")
                if bucket:
                    break
        out[s] = bucket if bucket else "UNKNOWN"
    return out


def get_factor_data(stock_list, date):
    if len(stock_list) == 0:
        return pd.DataFrame()
    df = pd.DataFrame(index=stock_list)
    for fac_chunk in chunks(BASE_FACTOR_COLS, 20):
        try:
            factor_data = get_factor_values(securities=stock_list, factors=fac_chunk, count=1, end_date=date)
        except Exception as err:
            print("factor chunk failed", fac_chunk[:2], err)
            factor_data = None
        for fac in fac_chunk:
            try:
                if factor_data is not None and fac in factor_data:
                    df[fac] = factor_data[fac].iloc[0, :]
                else:
                    df[fac] = np.nan
            except Exception:
                df[fac] = np.nan
    return df


def calc_ret(close_mat, days):
    if close_mat is None or close_mat.empty or len(close_mat) <= days:
        return pd.Series(dtype=float)
    return close_mat.iloc[-1] / close_mat.iloc[-days - 1] - 1


def calc_up_day_ratio(ret_mat, days):
    if ret_mat is None or ret_mat.empty:
        return pd.Series(dtype=float)
    return (ret_mat.tail(days) > 0).mean()


def calc_new_low_distance(close_mat, days):
    if close_mat is None or close_mat.empty:
        return pd.Series(dtype=float)
    return close_mat.iloc[-1] / close_mat.tail(days).min() - 1


def get_price_path_and_liquidity_data(stock_list, date, lookback=121, chunk_size=160):
    cols = V4_PRICE_PATH_COLS + TRADE_LIQUIDITY_COLS[:9]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk, end_date=date, frequency="daily",
                fields=["close", "high", "low", "volume", "money", "paused"],
                count=lookback, skip_paused=False, fq="pre", panel=False, fill_paused=True,
            )
        except Exception as err:
            print("price/liquidity failed", date, len(stock_chunk), err)
            price_df = None
        if price_df is None or price_df.empty:
            out_all.append(out)
            continue
        for col in ["close", "high", "low", "volume", "money", "paused"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        volume_mat = price_df.pivot_table(index="time", columns="code", values="volume").sort_index()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        paused_mat = price_df.pivot_table(index="time", columns="code", values="paused").sort_index()
        ret_mat = close_mat.pct_change()
        last_close = close_mat.iloc[-1]
        ma20 = close_mat.tail(20).mean()
        ma60 = close_mat.tail(60).mean()
        money20 = money_mat.tail(20).mean()
        money60 = money_mat.tail(60).mean()
        volume20 = volume_mat.tail(20).mean()
        volume60 = volume_mat.tail(60).mean()
        out["px_ret_5"] = calc_ret(close_mat, 5)
        out["px_ret_20"] = calc_ret(close_mat, 20)
        out["px_ret_60"] = calc_ret(close_mat, 60)
        out["px_ret_120"] = calc_ret(close_mat, 120)
        out["px_close_to_ma20"] = last_close / ma20 - 1
        out["px_close_to_ma60"] = last_close / ma60 - 1
        out["px_ma20_to_ma60"] = ma20 / ma60 - 1
        out["px_volatility_20"] = ret_mat.tail(20).std()
        out["px_volatility_60"] = ret_mat.tail(60).std()
        out["px_drawdown_20"] = last_close / close_mat.tail(20).max() - 1
        out["px_drawdown_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_drawdown_120"] = last_close / close_mat.tail(120).max() - 1
        out["px_up_day_ratio_20"] = calc_up_day_ratio(ret_mat, 20)
        out["px_new_high_distance_60"] = last_close / close_mat.tail(60).max() - 1
        out["px_new_low_distance_60"] = calc_new_low_distance(close_mat, 60)
        out["px_skew_20"] = ret_mat.tail(20).skew()
        out["px_kurt_20"] = ret_mat.tail(20).kurt()
        out["liq_money_mean_20"] = money20
        out["liq_money_mean_60"] = money60
        out["liq_money_ratio_20_60"] = money20 / money60 - 1
        out["liq_volume_mean_20"] = volume20
        out["liq_volume_ratio_20_60"] = volume20 / volume60 - 1
        out["liq_amplitude_mean_20"] = (high_mat.tail(20) / low_mat.tail(20) - 1).mean()
        out["liq_amplitude_mean_60"] = (high_mat.tail(60) / low_mat.tail(60) - 1).mean()
        out["liq_paused_count_20"] = paused_mat.tail(20).fillna(0).sum()
        out["liq_paused_count_60"] = paused_mat.tail(60).fillna(0).sum()
        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, close_mat, high_mat, low_mat, volume_mat, money_mat, paused_mat, ret_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list) if out_all else pd.DataFrame(index=stock_list)


def get_limit_state_data(stock_list, date, lookback=20, chunk_size=160):
    cols = ["liq_low_money_days_20", "liq_limit_up_count_20", "liq_limit_down_count_20", "liq_one_price_limit_count_20"]
    out_all = []
    for stock_chunk in chunks(stock_list, chunk_size):
        out = pd.DataFrame(index=stock_chunk, columns=cols, dtype=float)
        try:
            price_df = get_price(
                stock_chunk, end_date=date, frequency="daily",
                fields=["close", "high", "low", "money", "paused", "high_limit", "low_limit"],
                count=lookback, skip_paused=False, fq=None, panel=False, fill_paused=True,
            )
        except Exception as err:
            print("limit state failed", date, len(stock_chunk), err)
            price_df = None
        if price_df is None or price_df.empty:
            out_all.append(out)
            continue
        for col in ["close", "high", "low", "money", "paused", "high_limit", "low_limit"]:
            if col not in price_df.columns:
                price_df[col] = np.nan
        price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
        money_mat = price_df.pivot_table(index="time", columns="code", values="money").sort_index()
        close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
        high_mat = price_df.pivot_table(index="time", columns="code", values="high").sort_index()
        low_mat = price_df.pivot_table(index="time", columns="code", values="low").sort_index()
        high_limit_mat = price_df.pivot_table(index="time", columns="code", values="high_limit").sort_index()
        low_limit_mat = price_df.pivot_table(index="time", columns="code", values="low_limit").sort_index()
        money_stack = money_mat.stack().dropna()
        money_q20 = money_stack.quantile(0.20) if len(money_stack) else np.nan
        out["liq_low_money_days_20"] = (money_mat.tail(20) < money_q20).sum() if not pd.isnull(money_q20) else np.nan
        limit_up = close_mat >= (high_limit_mat * 0.999)
        limit_down = close_mat <= (low_limit_mat * 1.001)
        one_price = (high_mat <= low_mat * 1.0001) & (limit_up | limit_down)
        out["liq_limit_up_count_20"] = limit_up.tail(20).sum()
        out["liq_limit_down_count_20"] = limit_down.tail(20).sum()
        out["liq_one_price_limit_count_20"] = one_price.tail(20).sum()
        out_all.append(out.replace([np.inf, -np.inf], np.nan))
        del price_df, money_mat, close_mat, high_mat, low_mat, high_limit_mat, low_limit_mat
        gc.collect()
    return pd.concat(out_all).reindex(index=stock_list) if out_all else pd.DataFrame(index=stock_list)


def get_forward_returns(stock_list, rebalance_date, next_date, benchmark=BENCHMARK):
    if len(stock_list) == 0:
        return pd.DataFrame(columns=["stock_return", "bench_return", "alpha_1m"])
    price_df = get_price(
        stock_list, start_date=rebalance_date, end_date=next_date, frequency="daily",
        fields=["close"], skip_paused=True, fq="pre", panel=False,
    )
    if price_df is None or price_df.empty:
        return pd.DataFrame(columns=["stock_return", "bench_return", "alpha_1m"])
    price_df["time"] = pd.to_datetime(price_df["time"]).dt.normalize()
    close_mat = price_df.pivot_table(index="time", columns="code", values="close").sort_index()
    if len(close_mat) < 2:
        return pd.DataFrame(columns=["stock_return", "bench_return", "alpha_1m"])
    # 沿用旧 V4 label 近似：用区间第二根到最后一根，避开调仓日 close。
    stock_ret = close_mat.iloc[-1] / close_mat.iloc[1] - 1
    bench_df = get_price(benchmark, start_date=rebalance_date, end_date=next_date, frequency="daily", fields=["close"], skip_paused=True, fq="pre")
    if bench_df is None or bench_df.empty or len(bench_df) < 2:
        bench_ret = np.nan
    else:
        bench_ret = bench_df["close"].iloc[-1] / bench_df["close"].iloc[1] - 1
    out = pd.DataFrame({"stock_return": stock_ret, "bench_return": bench_ret})
    out["alpha_1m"] = out["stock_return"] - out["bench_return"]
    return out


def get_market_context(date, benchmark=BENCHMARK, lookback=61):
    out = {"ctx_market_ret_20": np.nan, "ctx_market_ret_60": np.nan, "ctx_market_volatility_20": np.nan}
    try:
        df = get_price(benchmark, end_date=date, frequency="daily", fields=["close"], count=lookback, skip_paused=True, fq="pre")
    except Exception:
        df = None
    if df is None or df.empty or "close" not in df.columns:
        return out
    close = df["close"].dropna()
    if len(close) > 20:
        out["ctx_market_ret_20"] = close.iloc[-1] / close.iloc[-21] - 1
        out["ctx_market_volatility_20"] = close.pct_change().tail(20).std()
    if len(close) > 60:
        out["ctx_market_ret_60"] = close.iloc[-1] / close.iloc[-61] - 1
    return out


def attach_industry_context(factor_data, market_context):
    out = factor_data.copy()
    for col, value in market_context.items():
        out[col] = value
    for ret_col, ctx_col in [("px_ret_20", "ctx_industry_ret_20"), ("px_ret_60", "ctx_industry_ret_60")]:
        out[ctx_col] = out.groupby("industry_bucket")[ret_col].transform("mean")
    out["ctx_stock_minus_industry_ret_20"] = out["px_ret_20"] - out["ctx_industry_ret_20"]
    out["ctx_stock_minus_industry_ret_60"] = out["px_ret_60"] - out["ctx_industry_ret_60"]
    out["ctx_stock_rank_industry_ret_20"] = out.groupby("industry_bucket")["px_ret_20"].rank(pct=True)
    out["ctx_stock_rank_industry_volatility_20"] = out.groupby("industry_bucket")["px_volatility_20"].rank(pct=True)
    return out


def add_core_factor_temporal_features(df):
    out = df.copy().sort_values(["rebalance_date", "stock"]).reset_index(drop=True)
    for factor in CORE_TEMPORAL_FACTORS:
        if factor not in out.columns:
            continue
        rank_col = "tmp_{}_rank".format(factor)
        out[rank_col] = out.groupby("rebalance_date")[factor].rank(pct=True)
        g_stock = out.groupby("stock")[rank_col]
        for lag in [1, 3]:
            out["ts_{}_rank_chg_{}m".format(factor, lag)] = out[rank_col] - g_stock.shift(lag)
        out["ts_{}_rank_mean_3m".format(factor)] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
        out["ts_{}_rank_std_3m".format(factor)] = g_stock.transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
        rolling_mean_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
        rolling_std_6 = g_stock.transform(lambda s: s.shift(1).rolling(6, min_periods=3).std())
        out["ts_{}_rank_z_6m".format(factor)] = (out[rank_col] - rolling_mean_6) / rolling_std_6
        out = out.drop(columns=[rank_col])
    return out

print("feature builders ready")

In [ ]:
# =========================
# Cell 6: Build monthly dataset for one pool
# =========================
def build_pool_dataset(pool_config):
    pool_name = pool_config["name"]
    cache_path = os.path.join(CACHE_DIR, "{}_{}_{}.csv".format(pool_name, EXPERIMENT_START.replace("-", ""), EXPERIMENT_END_FOR_LABEL.replace("-", "")))
    if (not FORCE_REBUILD) and os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        for col in ["rebalance_date", "feature_date", "next_date"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col])
        print("loaded cache", pool_name, len(df), cache_path)
        return df

    date_list = get_period_date("M", EXPERIMENT_START, EXPERIMENT_END_FOR_LABEL)
    print("building", pool_name, "months=", max(0, len(date_list) - 1))
    all_rows = []
    for i, rebalance_date in enumerate(date_list[:-1]):
        next_date = date_list[i + 1]
        feature_date = get_previous_trade_date(rebalance_date)
        if feature_date is None:
            continue
        stock_list = get_prefiltered_pool(pool_config, feature_date)
        if len(stock_list) == 0:
            continue

        price_liq = get_price_path_and_liquidity_data(stock_list, feature_date)
        if bool(pool_config.get("liquidity_filter", False)):
            before = len(stock_list)
            ok = price_liq[
                (price_liq["liq_money_mean_20"] >= LIQ_MIN_MONEY_20) &
                (price_liq["liq_paused_count_20"].fillna(999) <= LIQ_MAX_PAUSED_20)
            ].index.tolist()
            stock_list = [s for s in stock_list if s in set(ok)]
            price_liq = price_liq.reindex(stock_list)
            print("  {} {} liquidity {}/{}".format(pool_name, feature_date, len(stock_list), before))

        if pool_config.get("smallcap_limit") is not None:
            before = len(stock_list)
            stock_list = filter_smallcap_by_market_cap(stock_list, feature_date, pool_config.get("smallcap_limit"))
            price_liq = price_liq.reindex(stock_list)
            print("  {} {} smallcap {}/{}".format(pool_name, feature_date, len(stock_list), before))

        if len(stock_list) < TOP_N_PORTFOLIO:
            continue

        jq_factor = get_factor_data(stock_list, feature_date)
        if jq_factor is None or jq_factor.empty:
            continue
        industry_map = get_industry_bucket_map(stock_list, feature_date)
        limit_data = get_limit_state_data(stock_list, feature_date)
        fwd = get_forward_returns(stock_list, rebalance_date, next_date, BENCHMARK)
        if fwd.empty:
            continue

        factor_data = jq_factor.join(price_liq, how="left").join(limit_data, how="left").join(fwd, how="left")
        factor_data["stock"] = factor_data.index
        factor_data["industry_bucket"] = factor_data["stock"].map(industry_map).fillna("UNKNOWN")
        factor_data = attach_industry_context(factor_data, get_market_context(feature_date, BENCHMARK))
        factor_data["pool"] = pool_name
        factor_data["rebalance_date"] = rebalance_date
        factor_data["feature_date"] = feature_date
        factor_data["next_date"] = next_date
        factor_data = factor_data.dropna(subset=["alpha_1m", "stock_return"]).copy()
        if len(factor_data) < TOP_N_PORTFOLIO:
            continue
        factor_data["alpha_rank_pct"] = factor_data["alpha_1m"].rank(pct=True, method="first")
        all_rows.append(factor_data.reset_index(drop=True))
        print("  built {}/{} {} rows={}".format(i + 1, max(1, len(date_list) - 1), rebalance_date, len(factor_data)))
        del jq_factor, price_liq, limit_data, fwd, factor_data
        gc.collect()

    if not all_rows:
        raise ValueError("no rows built for {}".format(pool_name))
    df = pd.concat(all_rows, ignore_index=True)
    df = add_core_factor_temporal_features(df)
    df.to_csv(cache_path, index=False)
    print("saved", pool_name, len(df), cache_path)
    return df

print("dataset builder ready")

In [ ]:
# =========================
# Cell 7: Load model and scoring logic
# =========================
def load_pickle_file(path):
    # JQ strategy/runtime exposes read_file; JQ research often also supports normal file IO.
    try:
        raw = read_file(path)
        return pickle.loads(raw)
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)


def load_model_bundle(path=MODEL_FILE):
    bundle = load_pickle_file(path)
    if not isinstance(bundle, dict):
        raise ValueError("model bundle should be dict")
    print("loaded model", path)
    print("objective=", bundle.get("objective"), "research_version=", bundle.get("research_version"))
    return bundle


def predict_one_model(model, X, feature_cols, fill_values):
    X2 = X.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    X2 = X2.fillna(pd.Series(fill_values)).fillna(0)
    best_iter = getattr(model, "best_iteration", None)
    if best_iter is not None and best_iter > 0:
        return np.asarray(model.predict(X2[feature_cols], num_iteration=best_iter)).reshape(-1)
    return np.asarray(model.predict(X2[feature_cols])).reshape(-1)


def score_with_bundle(df, bundle):
    out = df.copy()
    overlay_mode = bundle.get("overlay_mode", "direct")
    base_cols = list(bundle.get("base_feature_cols", bundle.get("feature_cols", [])))
    base_fill = dict(bundle.get("base_fill_values", bundle.get("fill_values", {})))
    base_model = bundle.get("base_model", bundle.get("model"))
    if base_model is None or not base_cols:
        raise ValueError("bundle missing base model/feature columns")

    out["base_score"] = predict_one_model(base_model, out, base_cols, base_fill)
    out["base_score_z"] = out.groupby("rebalance_date")["base_score"].transform(zscore_series)

    resid_cols = list(bundle.get("residual_feature_cols", []))
    resid_model = bundle.get("residual_model", None)
    overlay_weight = float(bundle.get("overlay_weight", 0.15))
    if overlay_mode == "direct" or resid_model is None or len(resid_cols) == 0:
        out["residual_score"] = 0.0
        out["residual_score_z"] = 0.0
        out["final_score"] = out["base_score_z"]
    else:
        resid_fill = dict(bundle.get("residual_fill_values", {}))
        out["residual_score"] = predict_one_model(resid_model, out, resid_cols, resid_fill)
        out["residual_score_z"] = out.groupby("rebalance_date")["residual_score"].transform(zscore_series)
        out["final_score"] = out["base_score_z"] + overlay_weight * out["residual_score_z"]
    return out


def build_industry_neutral_targets(sorted_stocks, industry_map, target_num=TOP_N_PORTFOLIO, max_per_industry=2):
    selected = []
    counts = {}
    for s in sorted_stocks:
        ind = industry_map.get(s, "UNKNOWN")
        if counts.get(ind, 0) >= max_per_industry:
            continue
        selected.append(s)
        counts[ind] = counts.get(ind, 0) + 1
        if len(selected) >= target_num:
            break
    if len(selected) < target_num:
        for s in sorted_stocks:
            if s not in selected:
                selected.append(s)
            if len(selected) >= target_num:
                break
    return selected


def select_month_targets(month_df, top_n_candidates=TOP_N_CANDIDATES, target_num=TOP_N_PORTFOLIO):
    d = month_df.dropna(subset=["final_score", "base_score_z"]).copy()
    if d.empty:
        return []
    candidate_count = min(top_n_candidates, len(d))
    cand = d.nlargest(candidate_count, "base_score_z").sort_values("final_score", ascending=False)
    industry_map = cand.set_index("stock")["industry_bucket"].fillna("UNKNOWN").to_dict()
    max_per_ind = max(1, int(np.floor(target_num * INDUSTRY_CAP_RATIO)))
    return build_industry_neutral_targets(cand["stock"].tolist(), industry_map, target_num, max_per_ind)

bundle = load_model_bundle(MODEL_FILE)
print("model scoring ready")

In [ ]:
# =========================
# Cell 8: Run cross-universe experiment
# =========================
all_monthly = []
all_scored = []

for cfg in POOL_CONFIGS:
    pool_name = cfg["name"]
    df = build_pool_dataset(cfg)
    scored = score_with_bundle(df, bundle)
    all_scored.append(scored)

    prev_targets = None
    for dt, m in scored.groupby("rebalance_date"):
        m = m.copy()
        targets = select_month_targets(m)
        target_df = m[m["stock"].isin(targets)].copy()
        rank_ic = safe_rank_ic(m["final_score"], m["alpha_1m"])
        top30 = m.nlargest(min(TOP_N_CANDIDATES, len(m)), "base_score_z")
        turnover = np.nan
        if prev_targets is not None and len(targets) > 0:
            overlap = len(set(targets) & set(prev_targets))
            turnover = 1.0 - overlap / float(max(len(targets), 1))
        prev_targets = list(targets)
        all_monthly.append({
            "pool": pool_name,
            "rebalance_date": pd.Timestamp(dt),
            "next_date": pd.to_datetime(m["next_date"].iloc[0]),
            "universe_size": len(m),
            "rank_ic": rank_ic,
            "top10_count": len(target_df),
            "top10_return": target_df["stock_return"].mean() if len(target_df) else np.nan,
            "top10_alpha": target_df["alpha_1m"].mean() if len(target_df) else np.nan,
            "top30_alpha": top30["alpha_1m"].mean() if len(top30) else np.nan,
            "universe_alpha_mean": m["alpha_1m"].mean(),
            "bench_return": m["bench_return"].dropna().iloc[0] if m["bench_return"].notna().any() else np.nan,
            "top10_avg_money20": target_df["liq_money_mean_20"].mean() if len(target_df) else np.nan,
            "top10_min_money20": target_df["liq_money_mean_20"].min() if len(target_df) else np.nan,
            "top10_avg_mkt_liq_factor": target_df["liquidity"].mean() if "liquidity" in target_df.columns and len(target_df) else np.nan,
            "turnover": turnover,
            "targets": ",".join(targets),
        })
    print("finished", pool_name)

monthly = pd.DataFrame(all_monthly).sort_values(["pool", "rebalance_date"]).reset_index(drop=True)
scored_all = pd.concat(all_scored, ignore_index=True)
monthly.to_csv(os.path.join(CACHE_DIR, "universe_pool_monthly_results.csv"), index=False)
print(monthly.tail())

In [ ]:
# =========================
# Cell 9: Summary metrics
# =========================
def summarize_one(g):
    g = g.sort_values("rebalance_date").copy()
    n = len(g)
    r = g["top10_return"].astype(float)
    a = g["top10_alpha"].astype(float)
    out = {
        "months": n,
        "start": str(g["rebalance_date"].min().date()) if n else "",
        "end": str(g["rebalance_date"].max().date()) if n else "",
        "avg_universe_size": g["universe_size"].mean(),
        "ann_return": (1 + r.dropna()).prod() ** (12 / max(1, r.dropna().shape[0])) - 1 if r.notna().any() else np.nan,
        "ann_alpha": (1 + a.dropna()).prod() ** (12 / max(1, a.dropna().shape[0])) - 1 if a.notna().any() else np.nan,
        "monthly_return_mean": r.mean(),
        "monthly_alpha_mean": a.mean(),
        "monthly_alpha_vol": a.std(),
        "alpha_ir": a.mean() / a.std() * np.sqrt(12) if a.std() and a.std() > 0 else np.nan,
        "return_sharpe_like": r.mean() / r.std() * np.sqrt(12) if r.std() and r.std() > 0 else np.nan,
        "max_drawdown_return": max_drawdown(r),
        "max_drawdown_alpha": max_drawdown(a),
        "alpha_win_rate": (a > 0).mean(),
        "avg_rank_ic": g["rank_ic"].mean(),
        "rank_ic_ir": g["rank_ic"].mean() / g["rank_ic"].std() if g["rank_ic"].std() and g["rank_ic"].std() > 0 else np.nan,
        "avg_turnover": g["turnover"].mean(),
        "top10_avg_money20": g["top10_avg_money20"].mean(),
        "top10_min_money20_median": g["top10_min_money20"].median(),
    }
    return pd.Series(out)

summary = monthly.groupby("pool").apply(summarize_one).reset_index()
summary = summary.sort_values(["ann_alpha", "alpha_ir", "avg_rank_ic"], ascending=False)
summary.to_csv(os.path.join(CACHE_DIR, "universe_pool_summary.csv"), index=False)
summary

In [ ]:
# =========================
# Cell 10: Charts
# =========================
import matplotlib.pyplot as plt

plot_df = monthly.copy()
plot_df["nav_return"] = plot_df.groupby("pool")["top10_return"].transform(lambda s: (1 + s.fillna(0)).cumprod())
plot_df["nav_alpha"] = plot_df.groupby("pool")["top10_alpha"].transform(lambda s: (1 + s.fillna(0)).cumprod())

plt.figure(figsize=(13, 5))
for pool, g in plot_df.groupby("pool"):
    plt.plot(g["rebalance_date"], g["nav_alpha"], label=pool)
plt.title("Top10 monthly alpha NAV vs CSI800 benchmark")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(13, 5))
for pool, g in plot_df.groupby("pool"):
    plt.plot(g["rebalance_date"], g["rank_ic"].rolling(6, min_periods=1).mean(), label=pool)
plt.title("6-month rolling RankIC")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
summary.sort_values("ann_alpha").plot.barh(x="pool", y="ann_alpha", legend=False, figsize=(10, 5))
plt.title("Annualized alpha by pool")
plt.grid(True, axis="x", alpha=0.3)
plt.show()

In [ ]:
# =========================
# Cell 11: Diagnostics by year and selected names
# =========================
monthly["year"] = monthly["rebalance_date"].dt.year
by_year = monthly.groupby(["pool", "year"]).agg(
    months=("top10_alpha", "count"),
    ann_alpha=("top10_alpha", lambda s: (1 + s.dropna()).prod() ** (12 / max(1, len(s.dropna()))) - 1),
    avg_rank_ic=("rank_ic", "mean"),
    win_rate=("top10_alpha", lambda s: (s > 0).mean()),
    avg_turnover=("turnover", "mean"),
    avg_money20=("top10_avg_money20", "mean"),
).reset_index()

print("By-year diagnostics")
display(by_year)

print("Latest month targets")
latest = monthly.sort_values("rebalance_date").groupby("pool").tail(1)
display(latest[["pool", "rebalance_date", "top10_alpha", "rank_ic", "top10_avg_money20", "targets"]])

## 如何解读

优先看四件事：

1. `ann_alpha` 和 `alpha_ir`：是否真的比中证800原池更强。
2. `avg_rank_ic` 和分年表现：模型排序能力是否稳定，不要只看一两个月组合收益。
3. `top10_avg_money20` / `top10_min_money20_median`：扩池收益是不是来自太差的流动性。
4. `avg_turnover`：如果扩池带来很高换手，真实成本会吃掉一部分结果。

我的建议判定规则：

- 如果 `CSI800_PLUS_1000_LIQ` 的年化 alpha、IC、分年稳定性明显优于 `CSI800`，且流动性没有显著恶化，它就是下一轮重训优先池。
- 如果 `ALL_A_LIQ` 最高但回撤、换手或最低成交额明显恶化，先不要直接实盘化，应改成更强流动性门槛或分市值分层建模。
- 如果 `CSI1000` 单独表现好但波动大，说明 alpha 可能来自中小盘段，下一步应做 `CSI800 core + CSI1000 satellite` 或分层模型。